In [ ]:
CASE STUDY 1 : 
# Hospital Readmission Prediction
# Logistic Regression with L2 Regularization

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# ---------------------------------------------------------
# 1. CREATE / LOAD DATASET
# ---------------------------------------------------------
# If you have a CSV file, replace this section with:
# df = pd.read_csv("hospital_readmission.csv")

np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "age": np.random.randint(18, 90, n),
    "heart_rate": np.random.randint(55, 130, n),
    "systolic_bp": np.random.randint(90, 180, n),
    "glucose": np.random.randint(70, 300, n),
    "prior_visits": np.random.randint(0, 10, n),
    "diagnosis_code": np.random.randint(0, 6, n)
})

# Create target variable:
# 1 = readmitted within 30 days
# 0 = not readmitted

risk_score = (
    0.03 * df["age"]
    + 0.015 * df["heart_rate"]
    - 0.01 * df["systolic_bp"]
    + 0.01 * df["glucose"]
    + 0.35 * df["prior_visits"]
    + 0.5 * df["diagnosis_code"]
)

probability = 1 / (1 + np.exp(-(
    (risk_score - risk_score.mean()) / risk_score.std()
)))

df["readmitted_30_days"] = (
    np.random.random(n) < probability
).astype(int)


# ---------------------------------------------------------
# 2. CHECK DATA
# ---------------------------------------------------------

print("First 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["readmitted_30_days"].value_counts())


# ---------------------------------------------------------
# 3. DEFINE FEATURES AND TARGET
# ---------------------------------------------------------

X = df.drop("readmitted_30_days", axis=1)
y = df["readmitted_30_days"]


# ---------------------------------------------------------
# 4. TRAIN-TEST SPLIT
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ---------------------------------------------------------
# 5. FEATURE SCALING
# ---------------------------------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ---------------------------------------------------------
# 6. LOGISTIC REGRESSION
# ---------------------------------------------------------
# penalty="l2" means L2 regularization.
# C controls the strength of regularization.
# Smaller C = stronger regularization.

model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    random_state=42
)

model.fit(X_train, y_train)


# ---------------------------------------------------------
# 7. PREDICTIONS
# ---------------------------------------------------------

y_pred = model.predict(X_test)

# Probability of 30-day readmission
y_prob = model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# 8. ROC-AUC
# ---------------------------------------------------------

roc_auc = roc_auc_score(y_test, y_prob)

print("\nROC-AUC Score:", roc_auc)


# ---------------------------------------------------------
# 9. CONFUSION MATRIX
# ---------------------------------------------------------

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


# ---------------------------------------------------------
# 10. CLASSIFICATION REPORT
# ---------------------------------------------------------

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# ---------------------------------------------------------
# 11. MODEL COEFFICIENTS
# ---------------------------------------------------------

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

feature_importance["Absolute_Coefficient"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    by="Absolute_Coefficient",
    ascending=False
)

print("\nFeature coefficients:")
print(feature_importance)


# ---------------------------------------------------------
# 12. ROC CURVE
# ---------------------------------------------------------

import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Hospital Readmission Prediction")
plt.legend()
plt.grid()

plt.show()